
# B 방식: 제품 × 1명 프롬프트 생성 (v5)
입력
- `persona_attributes_weighted.jsonl` (가중치+meta 포함)
- `product_info_preprocessed.jsonl` (prompt_block 포함)

출력
- `prompts_B.jsonl`
- `prompts_B_preview.json` (샘플 확인용)


In [2]:

# =============================
# 0) CONFIG
# =============================
from pathlib import Path

PERSONA_JSONL = Path("persona_attributes_weighted.jsonl")
PRODUCT_JSONL = Path("product_info_preprocessed.jsonl")

OUT_JSONL     = Path("prompts_B.jsonl")
OUT_PREVIEW   = Path("prompts_B_preview.json")

# 실행 범위 제한(선택): None이면 전체 사용
LIMIT_PRODUCTS = None   # 예: 3
LIMIT_PERSONAS = None   # 예: 100

# 출력 포맷 옵션
ATTR_LIMIT = 60  # 프롬프트에 표시할 속성 최대 개수
print("CONFIG loaded.")

CONFIG loaded.


In [3]:

# =============================
# 1) Load data
# =============================
import json

# Personas
personas = []
with open(PERSONA_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            personas.append(json.loads(line))

# Products
products = []
with open(PRODUCT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            products.append(json.loads(line))

if LIMIT_PRODUCTS: products = products[:LIMIT_PRODUCTS]
if LIMIT_PERSONAS: personas = personas[:LIMIT_PERSONAS]

print("Loaded:", len(personas), "personas /", len(products), "products")
print("Sample product:", products[0] if products else None)

Loaded: 363 personas / 15 products
Sample product: {'product_id': None, 'product_name': '덴마크 하이그릭요거트 400g', 'category': '우유류 > 발효유 > 호상-중대용량', 'features': ['건강식품', '고단백', '고소한맛', '높은 만족도'], 'targeted_consumer': ['유당불내증'], 'release_info': '2025년 2월 출시', 'price_text': '3,980원', 'advertise_info': '광고/프로모션: 2025년 6-7월, 일반인 광고', 'ad_model': None, 'prompt_block': '- 제품명: 덴마크 하이그릭요거트 400g\n- 카테고리: 우유류 > 발효유 > 호상-중대용량\n- 주요 특징: 건강식품, 고단백, 고소한맛, 높은 만족도\n- 타깃: 유당불내증\n- 출시일: 2025년 2월 출시\n- 기준 가격대: 3,980원\n- 광고/프로모션: 2025년 6-7월, 일반인 광고'}


In [4]:

# =============================
# 2) Helpers
# =============================
from typing import Dict, Any

def format_attributes_for_prompt(attrs: Dict[str, Any], limit:int=60) -> str:
    lines = []
    # 정렬: weight 내림차순(가독성↑)
    ordered = sorted(attrs.items(), key=lambda kv: kv[1].get("weight", 0.0), reverse=True)
    for k, vw in ordered[:limit]:
        v = vw.get("value", None)
        w = vw.get("weight", 0.0)
        v_str = "None" if v is None else str(v)
        lines.append(f"- {k}: {v_str} (w={w:.3f})")
    return "\n".join(lines)

def build_product_block(p: Dict[str, Any]) -> str:
    # prompt_block이 있으면 그대로 사용
    if p.get("prompt_block"):
        return p["prompt_block"]
    # 백업 구성
    lines = []
    if p.get("product_id"): lines.append(f"- product_id: {p['product_id']}")
    if p.get("product_name"): lines.append(f"- 제품명: {p['product_name']}")
    if p.get("category"): lines.append(f"- 카테고리: {p['category']}")
    if p.get("features"): lines.append(f"- 주요 특징: {', '.join(p['features'])}")
    if p.get("targeted_consumer"): lines.append(f"- 타깃: {', '.join(p['targeted_consumer'])}")
    if p.get("release_info"): lines.append(f"- 출시일: {p['release_info']}")
    if p.get("price_text"): lines.append(f"- 기준 가격대: {p['price_text']}")
    if p.get("ad_model"): lines.append(f"- 광고모델: {p['ad_model']}")
    if p.get("advertise_info"): lines.append(f"- {p['advertise_info']}")
    return "\n".join(lines)

def build_single_prompt(product: Dict[str, Any], persona: Dict[str, Any], attr_limit:int=60) -> str:
    product_block = build_product_block(product)
    meta = persona.get("meta", {}) or {}
    cluster = meta.get("cluster", "")
    label = meta.get("label", "")
    desc = meta.get("desc", meta.get("Description",""))

    cluster_block = f"""
[클러스터 컨텍스트]
- cluster: {cluster}
- label: {label}
- desc: {desc}
""".strip() if (cluster or label or desc) else "[클러스터 컨텍스트]\n- N/A"

    return f"""
[역할]
당신은 한국 소비자 데이터 분석가이자 마케팅 전문가입니다.
아래의 "제품 정보"와 "페르소나"를 바탕으로,
이 페르소나가 해당 제품의 구매자로서 성립하는 **싱글 턴** 페르소나 JSON을 생성하세요.

[제품 정보]
{product_block}

[페르소나]
- id: {persona.get('persona_key','')}
- 속성(가중치 합=1):
{format_attributes_for_prompt(persona.get('attributes', {}), limit=attr_limit)}

{cluster_block}

[규칙]
- '클러스터 컨텍스트'는 배경 지침으로만 사용합니다. 속성 가중치(합=1)와 충돌 시 '속성 가중치'를 우선합니다.
- 2024-07 ~ 2025-06 월별로 구매확률(prob 0~1)과 예상수량(qty 정수)을 제시합니다.
- 추석/설, 광고/프로모션/계절성을 반영합니다.
- **반드시 아래 JSON 스키마를 출력**하고, 불필요한 설명 문장은 출력하지 마세요.

[출력 스키마(JSON)]
{{
  "persona_id": "p_{{product_id_or_name}}_{persona.get('persona_key','')}",
  "product_name": "{product.get('product_name','')}",
  "product_id": "{product.get('product_id','')}",
  "segment_ref": "{persona.get('persona_key','')}",
  "attributes": {{ "{{속성명}}": {{"value": "<값>", "weight": <0~1> }}, "...": "..." }},
  "purchase_pattern": {{
    "avg_purchase_prob": <0~1>,
    "avg_purchase_qty": <int>,
    "seasonality": {{"추석": "+x%", "설": "+y%"}},
    "promotion_effect": "광고/프로모션 노출 시 +z%"
  }},
  "forecast_12mo": {{
    "2024-07": {{"prob": <0~1>, "qty": <int>}},
    "...": {{}}, 
    "2025-06": {{"prob": <0~1>, "qty": <int>}}
  }}
}}
""".strip()

In [5]:

# =============================
# 3) Build & save
# =============================
import json
from pathlib import Path

records = []
for prod in products:
    for persona in personas:
        pid_or_name = prod.get("product_id") or (prod.get("product_name","") or "").replace(" ", "_")
        prompt_text = build_single_prompt({**prod, "product_id_or_name": pid_or_name}, persona, attr_limit=ATTR_LIMIT)
        rec = {
            "product": {"product_id_or_name": pid_or_name, "product_name": prod.get("product_name")},
            "persona": {"persona_key": persona.get("persona_key")},
            "prompt": prompt_text
        }
        records.append(rec)

with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

Path(OUT_PREVIEW).write_text(json.dumps(records[:3], ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", OUT_JSONL, "size=", Path(OUT_JSONL).stat().st_size, "bytes")
len(records)

Saved: prompts_B.jsonl size= 16049253 bytes


5445